In [5]:
from pathlib import Path

from src.annotation import load_annotations
from src.images import crop_normalized_bbox
from src.lmstudio import analyze_image, load_prompt, save_result

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data"
htr_prompt = load_prompt(ROOT / "prompts" / "htr-v1.txt")

models = {
    "qwen3-vl-8b": "qwen3-vl-8b-instruct-mlx",
    "qwen3-vl-30b": "qwen3-vl-30b-a3b-instruct-mlx",
}
text_types = {"main_text", "marginal_text", "handwritten_annotation"}

for model_folder, model in models.items():
    for source in sorted((DATA / "images").glob("*.tif")):
        annotation_path = DATA / "ground-truth" / f"{source.stem}.json"
        if not annotation_path.exists():
            print(f"Needs region annotations: {source.name}")
            continue

        document = load_annotations(annotation_path)
        if document.coordinate_system != "normalized-1000":
            raise ValueError(f"Unsupported coordinates: {annotation_path}")

        for region in document.regions:
            if region.region_type not in text_types:
                continue

            destination = (
                DATA / "results" / source.stem / model_folder
                / "htr" / f"{region.id}-htr-v1.json"
            )
            if destination.exists():
                print(f"Skipping existing: {destination.name} ({model_folder})")
                continue

            try:
                crop_path = DATA / "crops" / source.stem / f"{region.id}.png"
                crop_path.parent.mkdir(parents=True, exist_ok=True)
                crop = crop_normalized_bbox(source, region.bbox, padding=30)
                crop.save(crop_path)

                print(f"HTR: {source.stem} / {region.id} / {model}", flush=True)
                response = analyze_image(
                    crop_path,
                    htr_prompt,
                    model=model,
                    temperature=0.0,
                    max_tokens=4096,
                )
                if not isinstance(response.parsed.get("transcription"), str):
                    raise ValueError("Model response has no transcription string")

                save_result(
                    response,
                    destination,
                    task="htr",
                    prompt_version="htr-v1",
                )
            except Exception as exc:
                print(f"FAILED {source.stem} / {region.id}: {exc}", flush=True)

Needs region annotations: 00003-1345-AB009712.tif
Needs region annotations: 00003-747-AB009300.tif
Needs region annotations: 00007-97-AB008730.tif
Skipping existing: human-382299a4-htr-v1.json (qwen3-vl-8b)
HTR: 1280_AB010309_0005 / human-dbb419a1 / qwen3-vl-8b-instruct-mlx
HTR: 1280_AB010309_0005 / human-855bed91 / qwen3-vl-8b-instruct-mlx
HTR: 1280_AB010309_0005 / human-81f43a2d / qwen3-vl-8b-instruct-mlx
Needs region annotations: 436_AB008999_0003.tif
Needs region annotations: 670_AB009188_0003.tif
Needs region annotations: 00003-1345-AB009712.tif
Needs region annotations: 00003-747-AB009300.tif
Needs region annotations: 00007-97-AB008730.tif
Skipping existing: human-382299a4-htr-v1.json (qwen3-vl-30b)
HTR: 1280_AB010309_0005 / human-dbb419a1 / qwen3-vl-30b-a3b-instruct-mlx
HTR: 1280_AB010309_0005 / human-855bed91 / qwen3-vl-30b-a3b-instruct-mlx
HTR: 1280_AB010309_0005 / human-81f43a2d / qwen3-vl-30b-a3b-instruct-mlx
Needs region annotations: 436_AB008999_0003.tif
Needs region annot